In [1]:
import json
import pathlib
import re

import pandas as pd

from tqdm.auto import tqdm

from iupac_name_parser import IUPACNameParser

In [2]:
ROOT_DIR = pathlib.Path.cwd()
RAW_DATA_DIR = ROOT_DIR / "raw_data"
TREATED_DATA_DIR = ROOT_DIR / "treated_data"

ROTATION_RESULTS_DIR = TREATED_DATA_DIR / "rotation_results"
assert ROTATION_RESULTS_DIR.exists()

EXTRACTED_DIR = RAW_DATA_DIR / "extracted"
assert EXTRACTED_DIR.exists()

DATA_BLOCKS_DIR = TREATED_DATA_DIR / "data_blocks"
assert DATA_BLOCKS_DIR.exists()

COLLECTED_BOLD_BLOCKS_PATH = TREATED_DATA_DIR / "collected_bold_blocks.csv"
COLLECTED_ROTATION_RESULTS_PATH = TREATED_DATA_DIR / "collected_rotation_results.csv"
ROTATION_RESULTS_FROM_BOLD_TEXT_PATH = (
    TREATED_DATA_DIR / "rotation_results_from_bold_text.csv"
)

In [3]:
mol_parser = IUPACNameParser()

In [4]:
pd.set_option("display.max_columns", None)


def head(df, n=2):
    display(df.head(n))
    print(f"Contains {len(df)} rows")


def get_extended_block(
    block: dict, molecule: str, article_id: str, file_stem: str
) -> dict | None:
    try:
        name, smiles, appears_ambiguous, stereo_ignored = mol_parser.to_smiles(molecule)
    except Exception:
        return None
    return {
        "article_id": article_id,
        "file_stem": file_stem,
        **block,
        "molecule": name,
        "smiles": smiles,
        "appears_ambiguous": appears_ambiguous,
        "stereo_ignored": stereo_ignored,
    }

## Collect boldface text blocks

In [5]:
files = list(EXTRACTED_DIR.glob("**/*.json"))
articles_with_bold_blocks = []
for file in tqdm(files, desc="Processing bold block files"):
    if file.stat().st_size == 0:
        continue
    with open(file, "r") as f:
        blocks_from_file = json.load(f)
    for block in blocks_from_file:
        extended_block = get_extended_block(
            block, block["text"], file.parent.name, file.stem
        )
        if extended_block is None:
            continue
        articles_with_bold_blocks.append(extended_block)

bold_blocks_df = pd.DataFrame(articles_with_bold_blocks)
bold_blocks_df.to_csv(COLLECTED_BOLD_BLOCKS_PATH, index=False)
head(bold_blocks_df)

Processing bold block files:   0%|          | 0/20560 [00:00<?, ?it/s]

,article_id,file_stem,type,start,stop,text,molecule,smiles,appears_ambiguous,stereo_ignored
0,2057268,ja511728b_si_001,bold,4315,4370,"tetrahydro-2,5-methanobenzo[b]oxepine-4-carbox...","tetrahydro-2,5-methanobenzo[b]oxepine-4-carbox...",O1C2=C(C3C(CC1C3)C(=O)[O-])C=CC=C2,True,False
1,2057268,ja511728b_si_001,bold,5878,5999,"(-)-Methyl (2S,3R,4S,5S,10R)-5,10- dihydroxy-...","(-)-Methyl (2S,3R,4S,5S,10R)-5,10-dihydroxy-2,...",O[C@]12C3=C(O[C@]([C@H]([C@@H]1C(=O)OC)C1=CC=C...,False,True


Contains 384913 rows


## Collect LLM rotation results

In [6]:
SHORT_THRESHOLD_DISTANCE = 200
LONG_THRESHOLD_DISTANCE = 800

UNITS = r"(?:cm\s*-1|ppm)"
UNITS_PATTERN = re.compile(rf"\(?{UNITS}\)?")

POSITIVE_NUMBER = r"\d+(?:\.\d+)?"
NUMBER_OR_RANGE = rf"{POSITIVE_NUMBER}(?:\s*-\s*{POSITIVE_NUMBER})?"
INSIDE_PARENTHESES = r"\((?:[^()]+)\)"
NMR_PEAK = rf"{NUMBER_OR_RANGE}(?:\s*{INSIDE_PARENTHESES})?"
NMR_PEAK_LIST = rf"{NMR_PEAK}(?:[\s,]+{NMR_PEAK})*"
NMR_PATTERN = re.compile(
    rf"""
        (?:\d*[HCF]\s*)?               # Optional number and H/C/F notation
        NMR                            # NMR literal
        (?:\s*{INSIDE_PARENTHESES})+   # Parenthesized csv data (1 or more)
        \s*                            # Optional whitespace
        [δ=:\s]*                       # Optional NMR notation
        {NMR_PEAK_LIST}                # NMR peak list
    """,
    re.VERBOSE,
)

HRMS_PATTERN = re.compile(
    rf"HRMS(.{{1,100}})?found[\s:]+{POSITIVE_NUMBER}", re.DOTALL | re.IGNORECASE
)

CSV_NUMBERS = rf"{POSITIVE_NUMBER}(?:\s*,\s*{POSITIVE_NUMBER})+"
IR_PATTERN = re.compile(
    rf"(?:FT)?IR[^\d]{{1,20}}{CSV_NUMBERS}",
    re.IGNORECASE,
)

SEP_SPACE_PATTERN = re.compile(r"[,;\.][,;\.\s]+")


def remove_analysis_data(text: str) -> str:
    text = UNITS_PATTERN.sub("", text)
    text = NMR_PATTERN.sub("", text)
    text = HRMS_PATTERN.sub("", text)
    text = IR_PATTERN.sub("", text)
    text = SEP_SPACE_PATTERN.sub(" ", text)
    return text.strip()


def retrieve_excess_or_ratio(
    rotation_block: dict,
    before_rotation_block: bool = False,
    data_blocks: list[dict] = None,
) -> str:
    article_id = rotation_block["article_id"]
    file_stem = rotation_block["file_stem"]
    if data_blocks is None:
        data_block_path = DATA_BLOCKS_DIR / article_id / f"{file_stem}.json"
        with open(data_block_path, "r") as f:
            data_blocks = json.load(f)
    if before_rotation_block:
        data_blocks = reversed(data_blocks)
    for data_block in data_blocks:
        if before_rotation_block:
            delta = rotation_block["start"] - data_block["stop"]
        else:
            delta = data_block["start"] - rotation_block["stop"]
        if delta > 0 and data_block["type"] == "rotation":
            return "unknown"
        if delta > 0 and data_block["type"] in ["excess", "ratio"]:
            if delta < SHORT_THRESHOLD_DISTANCE:
                return data_block["text"]
            if delta < LONG_THRESHOLD_DISTANCE:
                text_file = EXTRACTED_DIR / article_id / f"{file_stem}.txt"
                with open(text_file, "r") as f:
                    text = f.read()
                if before_rotation_block:
                    in_between = text[data_block["stop"] : rotation_block["start"]]
                else:
                    in_between = text[rotation_block["stop"] : data_block["start"]]
                in_between = remove_analysis_data(in_between).strip()
                if len(in_between) < SHORT_THRESHOLD_DISTANCE:
                    return data_block["text"]
                return "unknown"
            return "unknown"
    return "unknown"

In [7]:
rotation_results = []
rotation_results_files = list(ROTATION_RESULTS_DIR.glob("*"))
for path in tqdm(rotation_results_files, desc="Processing articles"):
    if not (path.is_dir() and path.name.isdigit()):
        continue
    article_id = path.name
    for subdir in path.glob("*"):
        if not subdir.is_dir():
            continue
        stem = subdir.name
        for block_file in subdir.glob("*.json"):
            with open(block_file, "r") as f:
                block = json.load(f)
            if block["molecule"] == "unknown":
                continue
            del block["type"]
            extended_block = get_extended_block(
                block, block["molecule"], article_id, stem
            )
            if extended_block is None:
                continue
            if extended_block["excess_or_ratio"] == "unknown":
                extended_block["excess_or_ratio"] = (
                    retrieve_excess_or_ratio(extended_block)
                )
                extended_block["excess_or_ratio_source"] = "data block"
            else:
                extended_block["excess_or_ratio_source"] = "LLM"
            rotation_results.append(extended_block)

rotation_results_df = pd.DataFrame(rotation_results)
rotation_results_df.to_csv(COLLECTED_ROTATION_RESULTS_PATH, index=False)
head(rotation_results_df)

Processing articles:   0%|          | 0/8389 [00:00<?, ?it/s]

,article_id,file_stem,start,stop,text,molecule,excess_or_ratio,smiles,appears_ambiguous,stereo_ignored,excess_or_ratio_source
0,2057268,ja511728b_si_001,13951,13985,"[α]D 26 = -75.37° (c = 0.1, CHCl3)","(-)-Methyl (1R,2R,3S,3aR,8bS)-3a-(4-bromopheny...",88% ee,BrC1=CC=C(C=C1)[C@@]12OC3=C([C@@]1([C@@H]([C@@...,False,True,LLM
1,2057268,ja511728b_si_001,9137,9172,"[α]D 26 = -83.017° (c = 0.2, CHCl3)","(-)-Methyl (2S,3R,4S,5S,10R)-2-(4-Bromophenyl)...",99% ee,BrC1=CC=C(C=C1)[C@]12[C@H]([C@@H]([C@](C3=C(O1...,False,True,LLM


Contains 108547 rows


In [8]:
head(rotation_results_df[rotation_results_df["excess_or_ratio"] != "unknown"])

,article_id,file_stem,start,stop,text,molecule,excess_or_ratio,smiles,appears_ambiguous,stereo_ignored,excess_or_ratio_source
0,2057268,ja511728b_si_001,13951,13985,"[α]D 26 = -75.37° (c = 0.1, CHCl3)","(-)-Methyl (1R,2R,3S,3aR,8bS)-3a-(4-bromopheny...",88% ee,BrC1=CC=C(C=C1)[C@@]12OC3=C([C@@]1([C@@H]([C@@...,False,True,LLM
1,2057268,ja511728b_si_001,9137,9172,"[α]D 26 = -83.017° (c = 0.2, CHCl3)","(-)-Methyl (2S,3R,4S,5S,10R)-2-(4-Bromophenyl)...",99% ee,BrC1=CC=C(C=C1)[C@]12[C@H]([C@@H]([C@](C3=C(O1...,False,True,LLM


Contains 89959 rows


### Extract rotation results from bold text

In [9]:
def retrieve_molecule(
    rotation_block: dict, bold_blocks: list[dict], full_text: str
) -> str:
    for bold_block in reversed(bold_blocks):
        delta = rotation_block["start"] - bold_block["stop"]
        if delta > 0:
            if delta < SHORT_THRESHOLD_DISTANCE:
                return bold_block["text"]
            if delta < LONG_THRESHOLD_DISTANCE:
                in_between = full_text[bold_block["stop"] : rotation_block["start"]]
                in_between = remove_analysis_data(in_between).strip()
                if len(in_between) < SHORT_THRESHOLD_DISTANCE:
                    return bold_block["text"]
            return "unknown"
    return "unknown"

In [10]:
def read_bold_blocks(article_id: str, file_stem: str) -> list[dict] | None:
    bold_block_path = EXTRACTED_DIR / article_id / f"{file_stem}.json"
    if not bold_block_path.exists() or bold_block_path.stat().st_size == 0:
        return None
    with open(bold_block_path, "r") as f:
        return json.load(f)


def read_full_text(article_id: str, file_stem: str) -> str:
    text_file = EXTRACTED_DIR / article_id / f"{file_stem}.txt"
    with open(text_file, "r") as f:
        return f.read()


rotation_results_from_bold_text = []

data_block_files = list(DATA_BLOCKS_DIR.glob("**/*.json"))
for data_block_file in tqdm(data_block_files, desc="Processing data block files"):
    article_id = data_block_file.parent.name
    file_stem = data_block_file.stem

    bold_blocks = read_bold_blocks(article_id, file_stem)
    if bold_blocks is None:
        continue
    full_text = read_full_text(article_id, file_stem)

    with open(data_block_file, "r") as f:
        data_blocks = json.load(f)
    for data_block in data_blocks:
        if data_block["type"] != "rotation":
            continue
        molecule = retrieve_molecule(data_block, bold_blocks, full_text)
        if molecule == "unknown":
            continue
        data_block["excess_or_ratio"] = "unknown"
        extended_block = get_extended_block(data_block, molecule, article_id, file_stem)
        if extended_block is None:
            continue
        excess_or_ratio = retrieve_excess_or_ratio(
            extended_block, before_rotation_block=True, data_blocks=data_blocks
        )
        if excess_or_ratio == "unknown":
            excess_or_ratio = retrieve_excess_or_ratio(
                extended_block, before_rotation_block=False, data_blocks=data_blocks
            )
        del extended_block["type"]
        extended_block["excess_or_ratio"] = excess_or_ratio
        rotation_results_from_bold_text.append(extended_block)

rotation_results_from_bold_text_df = pd.DataFrame(rotation_results_from_bold_text)
rotation_results_from_bold_text_df.to_csv(
    ROTATION_RESULTS_FROM_BOLD_TEXT_PATH, index=False
)
head(rotation_results_from_bold_text_df)

Processing data block files:   0%|          | 0/12574 [00:00<?, ?it/s]

,article_id,file_stem,start,stop,text,excess_or_ratio,molecule,smiles,appears_ambiguous,stereo_ignored
0,2057268,ja511728b_si_001,13951,13985,"[α]D 26 = -75.37° (c = 0.1, CHCl3)",88% ee,"(-)-Methyl (1R,2R,3S,3aR,8bS)-3a-(4-bromopheny...",BrC1=CC=C(C=C1)[C@@]12OC3=C([C@@]1([C@@H]([C@@...,False,True
1,2386192,ol401951g_si_002,6406,6437,"[α]D 25 = +27.4 (EtOAc, c 0.49)",94% ee,"(3R,4R,5R)-1-benzyl-5-methyl-2-oxo-4-phenylpip...",C(C1=CC=CC=C1)N1C([C@H]([C@@H]([C@H](C1)C)C1=C...,False,False


Contains 53923 rows


In [11]:
head(
    rotation_results_from_bold_text_df[
        rotation_results_from_bold_text_df["excess_or_ratio"] != "unknown"
    ]
)

,article_id,file_stem,start,stop,text,excess_or_ratio,molecule,smiles,appears_ambiguous,stereo_ignored
0,2057268,ja511728b_si_001,13951,13985,"[α]D 26 = -75.37° (c = 0.1, CHCl3)",88% ee,"(-)-Methyl (1R,2R,3S,3aR,8bS)-3a-(4-bromopheny...",BrC1=CC=C(C=C1)[C@@]12OC3=C([C@@]1([C@@H]([C@@...,False,True
1,2386192,ol401951g_si_002,6406,6437,"[α]D 25 = +27.4 (EtOAc, c 0.49)",94% ee,"(3R,4R,5R)-1-benzyl-5-methyl-2-oxo-4-phenylpip...",C(C1=CC=CC=C1)N1C([C@H]([C@@H]([C@H](C1)C)C1=C...,False,False


Contains 42281 rows
